# Training for Infill Model
Imports, params, loading dataset

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("/notebooks/"))  # make src/ importable

import torch
from pathlib import Path
import json
from src.dataset import load_cached_paths, download_dataset
from src.model import build_model, load_model, append_epoch_to_config
from src.generation_trainer import train_generation
from src.tokenizer import (
    VOCAB_SIZE
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"Vocabulary size: {VOCAB_SIZE} (includes infill tokens)")

Using device: cuda
Vocabulary size: 463 (includes infill tokens)


In [2]:
# Infill Training Configuration
EPOCHS = 2000
BATCH_SIZE = 8
LR = 1e-3
MAX_FILES = 10000
# FIM_RATIO = 0.5  # 50% FIM examples, 50% standard examples

In [3]:
# Load dataset
paths = load_cached_paths()
if len(paths) < MAX_FILES:
    print(f"Downloading up to {MAX_FILES} scores...")
    download_dataset(log=print, progress=lambda i, n: None, max_files=MAX_FILES)
    paths = load_cached_paths()
print(f"Using {len(paths)} score files for infill training.")

Fetching file tree from GitHub API...
Found 1462 score files.


Downloading: 100%|██████████| 1462/1462 [00:00<00:00, 83454.54it/s]

Dataset cached: 1462 files in /storage/data/openscore/
Using 1462 score files for infill training.


<hr>

### Build model

In [4]:
# Build infill model with expanded vocabulary
MODEL_NAME = 'GEN_D512_H16_L8'

print(f"Building {MODEL_NAME} model...")
model, cfg = build_model(name=MODEL_NAME, override_cfg=True)#load_model(device=DEVICE)
print(f"Infill model parameters: {model.num_params():,}")
print(f"Vocab size: {model.cfg['vocab_size']}")

model_log = []
def log_fn(msg):
    print(msg)
    model_log.append(msg)
        
def on_epoch(epoch, train_loss, val_loss):
    print(f"  → epoch {epoch}  train={train_loss:.4f}  val={val_loss:.4f}")
    append_epoch_to_config(MODEL_NAME, cfg, train_loss, val_loss)

Building GEN_D512_H16_L8 model...
Infill model parameters: 20,425,728
Vocab size: 463


<hr>

### Train model

In [ ]:
stop_flag = [False]  # set stop_flag[0] = True to interrupt training
train_generation(
    name=MODEL_NAME,
    model=model,
    data_paths=paths[:MAX_FILES],
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    device=DEVICE,
    log=log_fn,
    on_epoch=on_epoch,
    stop_flag=stop_flag,
    # fim_ratio=FIM_RATIO,
    resume_optimizer=False
)

Created val split: 73 val / 1389 train paths.
Parsing scores for GEN_D512_H16_L8 training...


Parsing scores...:   0%|          | 5/1389 [00:02<12:33,  1.84it/s]/usr/local/lib/python3.11/dist-packages/music21/musicxml/xmlToM21.py:5465: MusicXMLWarning: Could not import wedge: Error in getting DynamicWedges
  self.setDirectionInDirectionType(mxSpecificDirectionTag,
Parsing scores...:   1%|          | 10/1389 [00:04<10:23,  2.21it/s]/usr/local/lib/python3.11/dist-packages/music21/musicxml/xmlToM21.py:5465: MusicXMLWarning: Could not import pedal: Error in getting PedalMark
  self.setDirectionInDirectionType(mxSpecificDirectionTag,
Parsing scores...:   1%|          | 13/1389 [00:08<23:00,  1.00s/it]/usr/local/lib/python3.11/dist-packages/music21/musicxml/xmlToM21.py:5465: MusicXMLWarning: Could not import wedge: Error in getting DynamicWedges
  self.setDirectionInDirectionType(mxSpecificDirectionTag,
Parsing scores...:   2%|▏         | 29/1389 [00:16<08:06,  2.80it/s]/usr/local/lib/python3.11/dist-packages/music21/musicxml/xmlToM21.py:5516: MusicXMLWarning: Line <dashes> stop with